In [38]:
# ===============================
# 📦 IMPORTS
# ===============================
import os
import pandas as pd
import numpy as np
import joblib
import copy

import mlflow
import mlflow.sklearn

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

from sklearn.metrics import mean_squared_error, r2_score, mean_squared_log_error

In [39]:
import mlflow

mlflow.set_tracking_uri("file:f:/streamlit_session/guvi_projects/smart_premium/mlruns")

mlflow.set_experiment("Insurance_Premium_Project")

print("✅ CLEAN FILE-BASED MLflow ACTIVE")

Traceback (most recent call last):
  File "f:\streamlit_session\mlenv\Lib\site-packages\mlflow\store\tracking\file_store.py", line 383, in search_experiments
    exp = self._get_experiment(exp_id, view_type)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "f:\streamlit_session\mlenv\Lib\site-packages\mlflow\store\tracking\file_store.py", line 481, in _get_experiment
    meta = FileStore._read_yaml(experiment_dir, FileStore.META_DATA_FILE_NAME)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "f:\streamlit_session\mlenv\Lib\site-packages\mlflow\store\tracking\file_store.py", line 1670, in _read_yaml
    return _read_helper(root, file_name, attempts_remaining=retries)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "f:\streamlit_session\mlenv\Lib\site-packages\mlflow\store\tracking\file_store.py", line 1663, in _read_helper
    result = read_yaml(root, file_name)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "f:\s

✅ CLEAN FILE-BASED MLflow ACTIVE


In [40]:

# ===============================
# 📊 LOAD DATA
# ===============================
df = pd.read_csv("data/processed/train_cleaned.csv")

df["log_premium"] = np.log1p(df["Premium Amount"])

X = df.drop(columns=["Premium Amount", "log_premium"], errors="ignore")
y = df["log_premium"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)



In [41]:
# ===============================
# 🔧 PREPROCESSING (FIXED)
# ===============================
num_cols = X.select_dtypes(include="number").columns
cat_cols = X.select_dtypes(exclude="number").columns

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])



In [42]:
# ===============================
# 🤖 MODELS
# ===============================
models = {
    "LinearRegression": LinearRegression(),

    "RandomForest": RandomForestRegressor(
        n_estimators=150,
        max_depth=15,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features="sqrt",
        n_jobs=-1,
        random_state=42
    ),

    "XGBoost": XGBRegressor(
        n_estimators=150,
        learning_rate=0.08,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1,
        n_jobs=-1,
        random_state=42
    )
}



In [43]:
# ===============================
# 🏋️ TRAINING LOOP
# ===============================
results = {}

best_model = None
best_score = float("inf")
best_model_name = None



for name, model in models.items():

    with mlflow.start_run(run_name=name):

        print(f"\n🚀 Training {name}...")

        
        # ✅ Create NEW preprocessor each time
        preprocessor_new = ColumnTransformer([
            ("num", num_pipeline, num_cols),
            ("cat", cat_pipeline, cat_cols)
        ])


        
        # ✅ Fresh pipeline (CRITICAL FIX)
        pipe = Pipeline([
            ("preprocessor", copy.deepcopy(preprocessor_new)),
            ("model", copy.deepcopy(model))
        ])


        # Train
                       
        

        pipe.fit(X_train, y_train)

        # Predict
        y_pred = pipe.predict(X_test)

        # Metrics
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        r2 = r2_score(y_test, y_pred)

        # SAFE RMSLE (avoids log issues)
        try:
            rmsle = np.sqrt(mean_squared_log_error(
                y_test, y_pred
            ))
        except:
            rmsle = np.nan

        # ===============================
        # LOG TO MLFLOW
        # ===============================
        mlflow.log_param("model_name", name)
        mlflow.log_metric("rmse", rmse)
        mlflow.log_metric("r2", r2)
        mlflow.log_metric("rmsle", rmsle)

        mlflow.sklearn.log_model(pipe, "model")

        print(f"✅ {name} logged")
        print(f"{name} → RMSE: {rmse:.4f}, R2: {r2:.4f}, RMSLE: {rmsle:.4f}")

        # Save results
        results[name] = (rmse, r2, rmsle)

        # Best model selection
        if not np.isnan(rmsle) and rmsle < best_score:
            best_score = rmsle
            best_model = pipe
            best_model_name = name





🚀 Training LinearRegression...


2026/05/08 17:05:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/08 17:05:03 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


✅ LinearRegression logged
LinearRegression → RMSE: 1.0896, R2: 0.0124, RMSLE: 0.1645

🚀 Training RandomForest...


2026/05/08 17:05:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/08 17:05:42 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


✅ RandomForest logged
RandomForest → RMSE: 1.0640, R2: 0.0583, RMSLE: 0.1610

🚀 Training XGBoost...


2026/05/08 17:05:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/08 17:05:50 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


✅ XGBoost logged
XGBoost → RMSE: 1.0545, R2: 0.0749, RMSLE: 0.1595


In [44]:
# Save best model
os.makedirs("model", exist_ok=True)
joblib.dump(best_model, "model/best_model.pkl")
print("\n✅ Best model saved successfully!")
print(f"🏆 Best Model: {best_model_name}")


# ===============================
# 📊 SAVE RESULTS CSV (ADD HERE ✅)
# ===============================
results_df = pd.DataFrame(results).T
results_df.columns = ["RMSE", "R2", "RMSLE"]
results_df.to_csv("model/model_results.csv")

print("\n✅ Results saved to model/model_results.csv")

# ===============================
# 📊 FINAL COMPARISON
# ===============================
print("\n📊 Final Model Comparison:\n")

results_df = pd.DataFrame(results).T
results_df.columns = ["RMSE", "R2", "RMSLE"]

print(results_df.sort_values("RMSLE"))



✅ Best model saved successfully!
🏆 Best Model: XGBoost

✅ Results saved to model/model_results.csv

📊 Final Model Comparison:

                      RMSE        R2     RMSLE
XGBoost           1.054530  0.074940  0.159454
RandomForest      1.063995  0.058259  0.160957
LinearRegression  1.089618  0.012355  0.164531
